# U-Net 2D — Test (không train, load trọng số từ Drive)

Chạy tuần tự: **Setup → Test**

In [ ]:
# ══════════════════════════════════════════════════════
# SETUP — U-Net 2D (Test Only, no training)
# ══════════════════════════════════════════════════════
%cd /kaggle/working
import os, gdown, torch

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ── Repo (default branch) ─────────────────────────────────────────────
if not os.path.exists('Prompt-Guided-XRay-Segmentation'):
    !git clone -q https://github.com/ThongLuc2k3/Prompt-Guided-XRay-Segmentation.git
else:
    print('[Skip] Repo đã tồn tại')

# ── Dataset ───────────────────────────────────────────────────────────
DS_ZIP  = '/kaggle/working/dataset_BTXRD.zip'
DS_DEST = '/kaggle/working/Prompt-Guided-XRay-Segmentation/dataset_BTXRD'
if not os.path.exists(DS_ZIP):
    gdown.download('https://drive.google.com/uc?id=1X1SY8T63pdBPIdrv_3P0gKVwoLxCa5sW',
                   DS_ZIP, quiet=False)
if not os.path.exists(DS_DEST):
    !unzip -oq {DS_ZIP} -d /kaggle/working/Prompt-Guided-XRay-Segmentation/
print('  ✅ Dataset ready')

%cd /kaggle/working/Prompt-Guided-XRay-Segmentation

# ── Checkpoint (từ Google Drive — cùng trọng số dùng trong test-subcat) ──
os.makedirs('checkpoints', exist_ok=True)
CKPT = 'checkpoints/unet_best.pth'
if not os.path.exists(CKPT):
    gdown.download('https://drive.google.com/uc?id=1hKyyw8WvN6WRyzjDE50upWI0NIDJbSMu',
                   CKPT, quiet=False)
print(f'  ✅ {CKPT}  ({os.path.getsize(CKPT)//1024} KB)')

!pip install -q tqdm opencv-python matplotlib scipy gdown
print('\n✅ Setup DONE!')


In [ ]:
# ══════════════════════════════════════════════════════
# TEST — U-Net 2D | ImgSize=512
# ══════════════════════════════════════════════════════
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from scipy.ndimage import binary_erosion, distance_transform_edt

from dataset import BTXRD_Dataset
from models.networks.unet_2D import unet_2D

DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_PATH = "checkpoints/unet_best.pth"
IMG_SIZE   = 512

def calc_hd95(pred, gt):
    pred, gt = pred.astype(bool), gt.astype(bool)
    if not pred.any() and not gt.any(): return 0.0
    if not pred.any() or  not gt.any(): return float(IMG_SIZE)
    pe = pred ^ binary_erosion(pred)
    ge = gt   ^ binary_erosion(gt)
    d1 = distance_transform_edt(~ge)[pe]
    d2 = distance_transform_edt(~pe)[ge]
    if not len(d1) or not len(d2): return float(IMG_SIZE)
    return float(max(np.percentile(d1, 95), np.percentile(d2, 95)))

def calc_cbl(pred_bin, gt_bin):
    if gt_bin.sum() == 0: return None
    ys, xs = np.where(gt_bin)
    gt_diag = np.sqrt((ys.max()-ys.min())**2 + (xs.max()-xs.min())**2) + 1e-6
    if pred_bin.sum() == 0: return 0.0
    yp, xp = np.where(pred_bin)
    d = np.sqrt((xp.mean()-xs.mean())**2 + (yp.mean()-ys.mean())**2)
    return float(np.clip(1.0 - d/gt_diag, 0.0, 1.0))

def get_centroid(binary_map):
    if binary_map.sum() == 0: return None, None
    ys, xs = np.where(binary_map)
    return float(xs.mean()), float(ys.mean())

# ── Load model ────────────────────────────────────────────────────────
model = unet_2D(in_channels=1, n_classes=1).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True))
model.eval()
print(f'✅ Model loaded  [{DEVICE}]  {os.path.getsize(MODEL_PATH)//1024} KB')

# ── Dataset ───────────────────────────────────────────────────────────
test_dataset = BTXRD_Dataset(
    image_dir="dataset_BTXRD/test/images",
    mask_dir="dataset_BTXRD/test/masks",
    img_size=IMG_SIZE, is_train=False
)
test_dataset.images = sorted(test_dataset.images)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

# ── Test loop ─────────────────────────────────────────────────────────
SHOW_INDEX = list(range(10))
all_dice, all_iou, all_pre, all_rec, all_hd95, all_cbl = [], [], [], [], [], []
smooth = 1e-5

with torch.no_grad():
    for idx, (images, masks) in enumerate(test_loader):
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        outputs = model(images)
        preds   = (torch.sigmoid(outputs) > 0.5).float()

        img_np = (images[0, 0].cpu().numpy() + 1) / 2.0
        gm = masks[0, 0].cpu().numpy()
        pm = preds[0, 0].cpu().numpy()

        tp = (pm * gm).sum(); fp = (pm * (1-gm)).sum(); fn = ((1-pm) * gm).sum()
        dice = (2*tp + smooth) / (2*tp + fp + fn + smooth)
        iou  = (tp + smooth)   / (tp + fp + fn + smooth)
        pre  = (tp + smooth)   / (tp + fp + smooth)
        rec  = (tp + smooth)   / (tp + fn + smooth)
        hd   = calc_hd95(pm.astype(bool), gm.astype(bool))
        cbl  = calc_cbl(pm.astype(bool), gm.astype(bool))

        all_dice.append(dice); all_iou.append(iou)
        all_pre.append(pre);   all_rec.append(rec)
        all_hd95.append(hd)
        if cbl is not None: all_cbl.append(cbl)

        if idx in SHOW_INDEX:
            cx_gt, cy_gt = get_centroid(gm)
            cx_p,  cy_p  = get_centroid(pm)
            fig, axes = plt.subplots(1, 4, figsize=(20, 5))
            fig.suptitle(f"U-Net 2D | Sample #{idx}", fontsize=14, fontweight='bold')

            axes[0].imshow(img_np, cmap='gray'); axes[0].set_title("Ảnh gốc", fontsize=11, fontweight='bold')

            axes[1].imshow(img_np, cmap='gray')
            green = np.zeros((*gm.shape, 4), dtype=np.float32); green[gm==1]=[0,1,0,0.3]
            axes[1].imshow(green)
            if gm.max() > 0: axes[1].contour(gm, [0.5], colors='lime', linewidths=1.5)
            if cx_gt is not None: axes[1].plot(cx_gt, cy_gt, 'o', color='lime', ms=8, markeredgecolor='black', label='Tâm GT')
            axes[1].legend(loc='lower right', fontsize=8); axes[1].set_title("Ground Truth", fontsize=11, fontweight='bold')

            axes[2].imshow(img_np, cmap='gray')
            red = np.zeros((*pm.shape, 4), dtype=np.float32); red[pm==1]=[1,0,0,0.3]
            axes[2].imshow(red)
            if pm.max() > 0: axes[2].contour(pm, [0.5], colors='red', linewidths=1.5)
            if cx_p is not None: axes[2].plot(cx_p, cy_p, 'o', color='red', ms=8, markeredgecolor='white', label='Tâm Pred')
            axes[2].legend(loc='lower right', fontsize=8); axes[2].set_title("Dự đoán (U-Net)", fontsize=11, fontweight='bold')

            axes[3].imshow(img_np, cmap='gray')
            if gm.max() > 0: axes[3].contour(gm, [0.5], colors='lime', linewidths=2)
            if pm.max() > 0: axes[3].contour(pm, [0.5], colors='red',  linewidths=2, linestyles='--')
            if cx_gt is not None: axes[3].plot(cx_gt, cy_gt, 'o', color='lime', ms=8, markeredgecolor='black')
            if cx_p  is not None: axes[3].plot(cx_p,  cy_p,  'o', color='red',  ms=8, markeredgecolor='white')
            if cx_gt is not None and cx_p is not None:
                axes[3].plot([cx_gt, cx_p], [cy_gt, cy_p], '--', color='yellow', lw=1.5)
            axes[3].set_title(f"Dice:{dice:.3f} IoU:{iou:.3f} HD95:{hd:.1f}px\nCBL:{cbl:.3f} Pre:{pre:.3f} Rec:{rec:.3f}",
                              fontsize=10, fontweight='bold')
            for ax in axes: ax.axis('off')
            plt.tight_layout(); plt.show()

# ── Final report ──────────────────────────────────────────────────────
print("\n" + "="*60)
print("FINAL TEST RESULTS — U-NET 2D | ImgSize=512")
print("="*60)
print(f"Mean Dice ↑      : {np.mean(all_dice):.4f}")
print(f"Mean IoU ↑       : {np.mean(all_iou):.4f}")
print(f"Mean Precision ↑ : {np.mean(all_pre):.4f}")
print(f"Mean Recall ↑    : {np.mean(all_rec):.4f}")
print(f"Mean HD95 ↓ (px) : {np.mean(all_hd95):.2f}")
print(f"Mean CBL ↑       : {np.mean(all_cbl):.4f}")
print(f"Total Samples    : {len(all_dice)}")
print("="*60)

import csv
os.makedirs("results", exist_ok=True)
csv_path = "results/unet2d_results.csv"
with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow(["model","dice","iou","precision","recall","hd95","cbl","n_samples"])
    w.writerow(["UNet2D",
                f"{np.mean(all_dice):.4f}", f"{np.mean(all_iou):.4f}",
                f"{np.mean(all_pre):.4f}",  f"{np.mean(all_rec):.4f}",
                f"{np.mean(all_hd95):.4f}", f"{np.mean(all_cbl):.4f}",
                len(all_dice)])
print(f"✅ CSV: {csv_path}")
